# Length Helper
Планировщик производства кабеля: изолирование → скрутка → намотка на барабаны.

**Использование:**
1. Заполните `input.xlsx` (заказы, склад, ёмкости барабанов, параметры).
2. Укажите пути к файлам в ячейке «Настройки».
3. Запустите все ячейки (Kernel → Restart & Run All).

In [1]:
# ─── Настройки ───────────────────────────────────────────────────────
# INPUT_FILE  = 'input.xlsx'
INPUT_FILE  = 'input26.xlsx'
OUTPUT_FILE = 'output26.xlsx'

In [2]:
import sys, importlib
# Перезагружаем модули при повторных запусках
for mod in ['models', 'parser', 'planner', 'exporter']:
    if mod in sys.modules:
        importlib.reload(sys.modules[mod])

from parser  import parse_input
from planner import plan
from exporter import export

In [3]:
# ─── Чтение входного файла ───────────────────────────────────────────
orders, raw_wires, insulated_cores, cable_stock, \
    core_drum_caps, cable_drum_caps, params = parse_input(INPUT_FILE)

print(f'Заказов:          {len(orders)}')
print(f'Барабанов ТПЖ:    {len(raw_wires)}')
print(f'Жил на складе:    {len(insulated_cores)}')
print(f'Кабеля на складе: {len(cable_stock)}')
print(f'Сечений жил:      {len(core_drum_caps)}')
print(f'Марок кабелей:    {len(cable_drum_caps)}')
print()
print(f'Макс. изолирование:   {params.max_insulation_run} м')
print(f'Макс. скрутка:        {params.max_twisting_run} м')
print(f'Мин. строит. длина:   {params.min_construction_length} м')
print(f'Спайка разрешена:     {params.allow_splicing}')

Заказов:          85
Барабанов ТПЖ:    1
Жил на складе:    0
Кабеля на складе: 0
Сечений жил:      4
Марок кабелей:    51

Макс. изолирование:   4500.0 м
Макс. скрутка:        2100.0 м
Мин. строит. длина:   450.0 м
Спайка разрешена:     False


In [4]:
# ─── Планирование ────────────────────────────────────────────────────
result = plan(
    orders, raw_wires, insulated_cores, cable_stock,
    core_drum_caps, cable_drum_caps, params
)

print(f'Партий скрутки:       {len(result.batches)}')
print(f'Прогонов изолирования:{len(result.insulation_runs)}')
print(f'Жил со склада:        {len(result.insulated_core_uses)}')
print(f'Кабеля со склада:     {len(result.cable_stock_uses)}')
print(f'Выходных барабанов:   {len(result.drum_assignments)}')
print()
if result.errors:
    print('=== ОШИБКИ ===')
    for e in result.errors:
        print(' ', e)
if result.warnings:
    print('=== ПРЕДУПРЕЖДЕНИЯ ===')
    for w in result.warnings:
        print(' ', w)

Партий скрутки:       50
Прогонов изолирования:84
Жил со склада:        0
Кабеля со склада:     0
Выходных барабанов:   96

=== ОШИБКИ ===
  ❌ КОСМОГЕР Вз-ВБШвнг(А)-LS 3х2,5ок(N,PE)-1  ТУ 27.32.13-004-32408375-2022 / Натуральный [LS]: нужно 473 м ТПЖ 2,5мк (прогон 465 м + заправка 5 м + допуск 3 м), доступно суммарно 0 м (ни одного куска достаточной длины).
  ❌ КОСМОГЕР Вз-ВБШвнг(А)-LS 3х2,5ок(N,PE)-1  ТУ 27.32.13-004-32408375-2022 / Желто-зеленый [LS]: нужно 473 м ТПЖ 2,5мк (прогон 465 м + заправка 5 м + допуск 3 м), доступно суммарно 0 м (ни одного куска достаточной длины).
  ❌ КОСМОГЕР Вз-ВБШвнг(А)-LS 3х2,5ок(N,PE)-1  ТУ 27.32.13-004-32408375-2022 / Синий [LS]: нужно 473 м ТПЖ 2,5мк (прогон 465 м + заправка 5 м + допуск 3 м), доступно суммарно 0 м (ни одного куска достаточной длины).
  ❌ КОСМОГЕР Вз-ВБШвнг(А)-LS 3х25мк(N,PE)-1  ТУ 27.32.13-004-32408375-2022 / Натуральный [LS]: нужно 1708 м ТПЖ 25ок (прогон 1700 м + заправка 5 м + допуск 3 м), доступно суммарно 0 м (ни одного куска д

In [5]:
# ─── Инструкция по скрутке ───────────────────────────────────────────
import pandas as pd
from collections import defaultdict

# Индексы: партия → [InsulationRun], [InsulatedCoreUse], [DrumAssignment]
ins_run_by_batch  = defaultdict(list)
ins_use_by_batch  = defaultdict(list)
drum_by_batch     = defaultdict(list)

for r in result.insulation_runs:
    ins_run_by_batch[r.for_batch_id].append(r)
for u in result.insulated_core_uses:
    ins_use_by_batch[u.for_batch_id].append(u)
for da in result.drum_assignments:
    if da.source == 'партия' and da.batch_id:
        drum_by_batch[da.batch_id].append(da)

for batch in result.batches:
    bid = batch.id
    segs_str = ' + '.join(str(int(s)) for s in batch.segments)
    fr_info  = f'  FR: {batch.fire_resistant}' if batch.fire_resistant else ''
    print('═' * 72)
    print(f'ПАРТИЯ {bid}:  {batch.cable_mark}')
    print(f'  Тип жилы: {batch.wire_key} | Материал: {batch.insulation_material}{fr_info}')
    print(f'  Отрезки:  {segs_str} = {int(batch.total_length)} м суммарно')
    print()

    # ── ЗАПРАВКА МАШИНЫ ───────────────────────────────────────────────
    print('  ЗАПРАВКА МАШИНЫ СКРУТКИ:')
    setup_rows = []
    for pos, color in enumerate(batch.colors, 1):
        run = next((r for r in ins_run_by_batch[bid] if r.color == color), None)
        use = next((u for u in ins_use_by_batch[bid] if u.color == color), None)

        if run:
            src_type = 'Прогон изолирования'
            src_name = f'{run.id}  (барабан/моток: {run.drum_type})'
            length_m = int(run.length)
        elif use:
            src_type = 'Готовая жила (склад)'
            src_name = use.source_name
            length_m = int(use.length)
        else:
            src_type = '— не определено —'
            src_name = ''
            length_m = 0

        setup_rows.append({
            'Поз.':    pos,
            'Цвет':    color,
            'Откуда':  src_type,
            'Источник': src_name,
            'Длина, м': length_m,
        })

    df_setup = pd.DataFrame(setup_rows).set_index('Поз.')
    display(df_setup)
    print()

    # ── ПОСЛЕДОВАТЕЛЬНОСТЬ ОТРЕЗКОВ И БАРАБАНЫ ────────────────────────
    print('  ПОСЛЕДОВАТЕЛЬНОСТЬ ОТРЕЗКОВ И ПРИЁМНЫЕ БАРАБАНЫ:')

    if not batch.segments:
        print('  (нет отрезков)\n')
        continue

    # Для каждого отрезка определяем, на какой барабан он идёт.
    # DrumAssignment.segments содержит длины в порядке добавления.
    # Копируем список сегментов каждого барабана и «тратим» их по очереди.
    drum_seg_queues = {
        da.id: list(da.segments) for da in drum_by_batch[bid]
    }
    drum_list = list(drum_by_batch[bid])  # упорядоченный список барабанов партии

    seq_rows = []
    cumulative = 0
    for step, seg in enumerate(batch.segments, 1):
        cumulative += seg
        # Найти барабан, у которого следующий неиспользованный сегмент совпадает
        matched_da = None
        for da in drum_list:
            q = drum_seg_queues[da.id]
            if q and abs(q[0] - seg) < 0.5:
                matched_da = da
                q.pop(0)
                break
        # Если не нашли точного совпадения — ищем любой барабан с этой длиной
        if matched_da is None:
            for da in drum_list:
                q = drum_seg_queues[da.id]
                for i, v in enumerate(q):
                    if abs(v - seg) < 0.5:
                        matched_da = da
                        q.pop(i)
                        break
                if matched_da:
                    break

        seq_rows.append({
            'Шаг':          step,
            'Длина, м':     int(seg),
            'Нарастающим, м': int(cumulative),
            'Барабан':      matched_da.id if matched_da else '—',
            'Тип барабана': matched_da.drum_type if matched_da else '—',
            'Ёмк., м':     int(matched_da.drum_capacity) if matched_da else 0,
        })

    df_seq = pd.DataFrame(seq_rows).set_index('Шаг')
    display(df_seq)
    print()

print('═' * 72)


════════════════════════════════════════════════════════════════════════
ПАРТИЯ ПС-001:  КОСМОГЕР Вз-ВБШвнг(А)-LS 3х2,5ок(N,PE)-1  ТУ 27.32.13-004-32408375-2022
  Тип жилы: 2,5мк | Материал: LS
  Отрезки:  465 = 465 м суммарно

  ЗАПРАВКА МАШИНЫ СКРУТКИ:


,Цвет,Откуда,Источник,"Длина, м"
Поз.,,,,
1,Натуральный,— не определено —,,0
2,Желто-зеленый,— не определено —,,0
3,Синий,— не определено —,,0



  ПОСЛЕДОВАТЕЛЬНОСТЬ ОТРЕЗКОВ И ПРИЁМНЫЕ БАРАБАНЫ:


,"Длина, м","Нарастающим, м",Барабан,Тип барабана,"Ёмк., м"
Шаг,,,,,
1,465,465,БК-001,№10,600



════════════════════════════════════════════════════════════════════════
ПАРТИЯ ПС-003:  КОСМОГЕР Вз-ВБШвнг(А)-LS 3х25мк(N,PE)-1  ТУ 27.32.13-004-32408375-2022
  Тип жилы: 25ок | Материал: LS
  Отрезки:  1700 = 1700 м суммарно

  ЗАПРАВКА МАШИНЫ СКРУТКИ:


...


════════════════════════════════════════════════════════════════════════
ПАРТИЯ ПС-097:  КОСМОСИЛ-НН РэВнг(А)-LS 3х6ок(N,PE)-1 ТУ 27.32.14-002-32408375-2022
  Тип жилы: 6мк | Материал: EPR
  Отрезки:  960 = 960 м суммарно

  ЗАПРАВКА МАШИНЫ СКРУТКИ:


,Цвет,Откуда,Источник,"Длина, м"
Поз.,,,,
1,Натуральный,— не определено —,,0
2,Желто-зеленый,— не определено —,,0
3,Синий,— не определено —,,0



  ПОСЛЕДОВАТЕЛЬНОСТЬ ОТРЕЗКОВ И ПРИЁМНЫЕ БАРАБАНЫ:


,"Длина, м","Нарастающим, м",Барабан,Тип барабана,"Ёмк., м"
Шаг,,,,,
1,960,960,—,—,0



════════════════════════════════════════════════════════════════════════


In [7]:
# ─── Инструкция по изолированию ──────────────────────────────────────
import pandas as pd

batch_mark  = {b.id: b.cable_mark  for b in result.batches}
batch_total = {b.id: b.total_length for b in result.batches}

# Начальные остатки ТПЖ-барабанов (из оригинального списка до планирования)
tpzh_avail = {rw.id: rw.length for rw in raw_wires}

# Начальные остатки изолированных жил на складе (до планирования)
inscore_avail = {ic.id: ic.length for ic in insulated_cores}

# Объединяем прогоны изолирования и использования со склада,
# сортируем по партии скрутки, затем по цвету — это рабочая последовательность
all_ops = sorted(
    [('ТПЖ', r) for r in result.insulation_runs] +
    [('СКЛ', u) for u in result.insulated_core_uses],
    key=lambda x: (x[1].for_batch_id, x[1].color)
)

rows = []
for n, (op_type, item) in enumerate(all_ops, 1):
    bid   = item.for_batch_id
    mark  = batch_mark.get(bid, '?')
    btot  = int(batch_total.get(bid, 0))

    if op_type == 'ТПЖ':
        run = item
        # raw_wire_consumed = длина прогона + потери на заправку + запас на обрезку торцов
        consumed = run.raw_wire_consumed if run.raw_wire_consumed > 0 else run.length
        before = round(tpzh_avail.get(run.source_id, 0))
        tpzh_avail[run.source_id] = tpzh_avail.get(run.source_id, 0) - consumed
        after = round(tpzh_avail[run.source_id])
        mat = run.insulation_material + (f'+{run.fire_resistant}' if run.fire_resistant else '')
        rows.append({
            '№':                  n,
            'Тип операции':       'Прогон (ТПЖ → изол.)',
            'Партия':             bid,
            'Марка кабеля':       mark,
            'Партия, м':          btot,
            'Цвет жилы':          run.color,
            'Тип жилы':           run.wire_key,
            'Материал/FR':        mat,
            'Источник':           run.source_name,
            'Остаток ДО, м':      before,
            'Длина прогона, м':   int(run.length),
            'Снять с барабана, м': int(consumed),
            'Остаток ПОСЛЕ, м':   after,
            'Принять на':         run.drum_type,
        })
    else:
        use = item
        before = round(inscore_avail.get(use.source_id, 0))
        inscore_avail[use.source_id] = use.remainder
        rows.append({
            '№':                  n,
            'Тип операции':       'Склад (готовая жила)',
            'Партия':             bid,
            'Марка кабеля':       mark,
            'Партия, м':          btot,
            'Цвет жилы':          use.color,
            'Тип жилы':           use.wire_key,
            'Материал/FR':        '(со склада)',
            'Источник':           use.source_name,
            'Остаток ДО, м':      before,
            'Длина прогона, м':   int(use.length),
            'Снять с барабана, м': '—',
            'Остаток ПОСЛЕ, м':   int(use.remainder),
            'Принять на':         '(уже готовая)',
        })

pd.DataFrame(rows)

,№,Тип операции,Партия,Марка кабеля,"Партия, м",Цвет жилы,Тип жилы,Материал/FR,Источник,"Остаток ДО, м","Длина прогона, м","Снять с барабана, м","Остаток ПОСЛЕ, м",Принять на
0,1,Прогон (ТПЖ → изол.),ПС-010,КОСМОГЕР Вз-ВБШвнг(А)-LS 4х25мк(N)-1 ТУ 27.32...,1804,Коричневый,25мк,LS,ТПЖ 25мк,500000,1804,1812,498188,Б-630
1,2,Прогон (ТПЖ → изол.),ПС-010,КОСМОГЕР Вз-ВБШвнг(А)-LS 4х25мк(N)-1 ТУ 27.32...,1804,Натуральный,25мк,LS,ТПЖ 25мк,498188,1804,1812,496376,Б-630
2,3,Прогон (ТПЖ → изол.),ПС-010,КОСМОГЕР Вз-ВБШвнг(А)-LS 4х25мк(N)-1 ТУ 27.32...,1804,Синий,25мк,LS,ТПЖ 25мк,496376,1804,1812,494564,Б-630
3,4,Прогон (ТПЖ → изол.),ПС-010,КОСМОГЕР Вз-ВБШвнг(А)-LS 4х25мк(N)-1 ТУ 27.32...,1804,Черный,25мк,LS,ТПЖ 25мк,494564,1804,1812,492752,Б-630
4,5,Прогон (ТПЖ → изол.),ПС-011,КОСМОГЕР Вз-ВБШвнг(А)-LS 4х25мк(N)-1 ТУ 27.32...,1804,Коричневый,25мк,LS,ТПЖ 25мк,492752,1804,1812,490940,Б-630
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
79,80,Прогон (ТПЖ → изол.),ПС-046,"КОСМОГЕР Вз-ВВГнг(А)-FRLS 5х25мк(N,PE)-1 ТУ 2...",1910,Желто-зеленый,25мк,LS+FR,ТПЖ 25мк,391793,1910,1918,389875,Б-630
80,81,Прогон (ТПЖ → изол.),ПС-046,"КОСМОГЕР Вз-ВВГнг(А)-FRLS 5х25мк(N,PE)-1 ТУ 2...",1910,Коричневый,25мк,LS+FR,ТПЖ 25мк,389875,1910,1918,387957,Б-630
81,82,Прогон (ТПЖ → изол.),ПС-046,"КОСМОГЕР Вз-ВВГнг(А)-FRLS 5х25мк(N,PE)-1 ТУ 2...",1910,Натуральный,25мк,LS+FR,ТПЖ 25мк,387957,1910,1918,386039,Б-630
82,83,Прогон (ТПЖ → изол.),ПС-046,"КОСМОГЕР Вз-ВВГнг(А)-FRLS 5х25мк(N,PE)-1 ТУ 2...",1910,Синий,25мк,LS+FR,ТПЖ 25мк,386039,1910,1918,384121,Б-630


In [8]:
# ─── Потребность в полуфабрикатах: ТПЖ и FR-жила ─────────────────────
#
# Логика:
#   • Стандартный прогон  → ТПЖ берётся как есть
#   • FR-прогон           → ТПЖ требует предварительной операции
#                           «нанесение огнестойкой обмотки» (или закупается
#                           ТПЖ FR как отдельный полуфабрикат)
#   • Ошибки планировщика → дефицит: жила не была выделена → требует заказа
# ─────────────────────────────────────────────────────────────────────
import pandas as pd
from collections import defaultdict

batch_mark = {b.id: b.cable_mark for b in result.batches}

# ── Потребность из успешно спланированных прогонов ───────────────────
plain_need: dict = defaultdict(float)   # wire_key → м стандарт
fr_need:    dict = defaultdict(float)   # wire_key → м с FR-обмоткой

for run in result.insulation_runs:
    if run.fire_resistant == 'FR':
        fr_need[run.wire_key] += run.length
    else:
        plain_need[run.wire_key] += run.length

# ── Дефицит: разбираем ошибки планировщика ───────────────────────────
# Ошибки вида «нужно X м ТПЖ Y, доступно Z м» — недостаток на складе
import re as _re

deficit_plain: dict = defaultdict(float)
deficit_fr:    dict = defaultdict(float)

for err in result.errors:
    m = _re.search(r'нужно\s+([\d]+)\s+м\s+ТПЖ\s+(\S+),', err)
    if not m:
        continue
    qty     = float(m.group(1))
    wk      = m.group(2)
    is_fr   = '+FR' in err or 'FR]' in err
    if is_fr:
        deficit_fr[wk]    += qty
    else:
        deficit_plain[wk] += qty

# ── Наличие ТПЖ на складе (исходное, до планирования) ────────────────
stock_by_wk: dict = defaultdict(float)
for rw in raw_wires:                    # оригинальный список — не изменён plan()
    stock_by_wk[rw.wire_key] += rw.length

all_wk = sorted(
    set(plain_need) | set(fr_need) | set(deficit_plain) | set(deficit_fr) | set(stock_by_wk)
)

# ══ Таблица 1: Потребность в ТПЖ ════════════════════════════════════
rows = []
for wk in all_wk:
    p     = plain_need.get(wk, 0.0)
    f     = fr_need.get(wk, 0.0)
    dp    = deficit_plain.get(wk, 0.0)
    df    = deficit_fr.get(wk, 0.0)
    total_need = p + f + dp + df
    stock      = stock_by_wk.get(wk, 0.0)
    order_qty  = dp + df                    # сколько нужно докупить/изготовить

    def _fmt(v): return int(v) if v > 0 else '—'
    rows.append({
        'Тип ТПЖ':              wk,
        'На складе, м':         int(stock),
        'Стандарт (план), м':   _fmt(p),
        'FR-обмотка (план), м': _fmt(f),
        'Стандарт (дефицит), м':_fmt(dp),
        'FR-обмотка (дефицит),м':_fmt(df),
        'Итого нужно, м':       int(total_need),
        'Действие':             f'⚠ Заказать {int(order_qty)} м' if order_qty > 0 else '✓ Достаточно',
    })

print('══════  ПОТРЕБНОСТЬ В ТПЖ (ГОЛАЯ ЖИЛА)  ══════\n')
display(pd.DataFrame(rows))

# ══ Таблица 2: Программа FR-обмотки ═════════════════════════════════
all_fr_runs = [r for r in result.insulation_runs if r.fire_resistant == 'FR']

if all_fr_runs or deficit_fr:
    print('\n══════  ПОЛУФАБРИКАТ: ЖИЛА С FR-ОБМОТКОЙ  ══════')
    print('Перед изолированием необходима операция нанесения огнестойкой обмотки (ленты).\n')

    if all_fr_runs:
        fr_rows = []
        for run in all_fr_runs:
            fr_rows.append({
                'Партия':             run.for_batch_id,
                'Марка кабеля':       batch_mark.get(run.for_batch_id, ''),
                'Цвет жилы':          run.color,
                'Тип ТПЖ':            run.wire_key,
                'Материал изол.':     run.insulation_material,
                'Источник (барабан)': run.source_name,
                'Длина прогона, м':   int(run.length),
                'Моток':              run.drum_type,
            })
        display(pd.DataFrame(fr_rows))

        # Сводка по барабанам ТПЖ, задействованным под FR
        print('\nБарабаны ТПЖ под FR-обмотку (нанести обмотку на отмеченные участки):')
        drum_fr: dict = defaultdict(float)
        for run in all_fr_runs:
            drum_fr[(run.wire_key, run.source_name)] += run.length
        for (wk, drum_name), meters in sorted(drum_fr.items()):
            print(f'  ТПЖ {wk:10s}  барабан {drum_name:20s}  → {int(meters):5d} м FR-обмотки')

    if deficit_fr:
        print('\n⚠  Дополнительно требуется (дефицит ТПЖ под FR, не покрытый складом):')
        for wk, qty in sorted(deficit_fr.items()):
            print(f'  ТПЖ {wk}: {int(qty)} м  → заказать или изготовить ТПЖ FR {wk}')
else:
    print('\nFR-жилы в текущей производственной программе отсутствуют.')

══════  ПОТРЕБНОСТЬ В ТПЖ (ГОЛАЯ ЖИЛА)  ══════



,Тип ТПЖ,"На складе, м","Стандарт (план), м","FR-обмотка (план), м","Стандарт (дефицит), м","FR-обмотка (дефицит),м","Итого нужно, м",Действие
0,1,0,—,—,10614,—,10614,⚠ Заказать 10614 м
1,2,0,—,—,10347,—,10347,⚠ Заказать 10347 м
2,25мк,500000,99600,17525,—,—,117125,✓ Достаточно



══════  ПОЛУФАБРИКАТ: ЖИЛА С FR-ОБМОТКОЙ  ══════
Перед изолированием необходима операция нанесения огнестойкой обмотки (ленты).



,Партия,Марка кабеля,Цвет жилы,Тип ТПЖ,Материал изол.,Источник (барабан),"Длина прогона, м",Моток
0,ПС-045,"КОСМОГЕР Вз-ВВГнг(А)-FRLS 5х25мк(N,PE)-1 ТУ 2...",Натуральный,25мк,LS,ТПЖ 25мк,1595,Б-630
1,ПС-045,"КОСМОГЕР Вз-ВВГнг(А)-FRLS 5х25мк(N,PE)-1 ТУ 2...",Желто-зеленый,25мк,LS,ТПЖ 25мк,1595,Б-630
2,ПС-045,"КОСМОГЕР Вз-ВВГнг(А)-FRLS 5х25мк(N,PE)-1 ТУ 2...",Синий,25мк,LS,ТПЖ 25мк,1595,Б-630
3,ПС-045,"КОСМОГЕР Вз-ВВГнг(А)-FRLS 5х25мк(N,PE)-1 ТУ 2...",Коричневый,25мк,LS,ТПЖ 25мк,1595,Б-630
4,ПС-045,"КОСМОГЕР Вз-ВВГнг(А)-FRLS 5х25мк(N,PE)-1 ТУ 2...",Черный,25мк,LS,ТПЖ 25мк,1595,Б-630
5,ПС-046,"КОСМОГЕР Вз-ВВГнг(А)-FRLS 5х25мк(N,PE)-1 ТУ 2...",Натуральный,25мк,LS,ТПЖ 25мк,1910,Б-630
6,ПС-046,"КОСМОГЕР Вз-ВВГнг(А)-FRLS 5х25мк(N,PE)-1 ТУ 2...",Желто-зеленый,25мк,LS,ТПЖ 25мк,1910,Б-630
7,ПС-046,"КОСМОГЕР Вз-ВВГнг(А)-FRLS 5х25мк(N,PE)-1 ТУ 2...",Синий,25мк,LS,ТПЖ 25мк,1910,Б-630
8,ПС-046,"КОСМОГЕР Вз-ВВГнг(А)-FRLS 5х25мк(N,PE)-1 ТУ 2...",Коричневый,25мк,LS,ТПЖ 25мк,1910,Б-630
9,ПС-046,"КОСМОГЕР Вз-ВВГнг(А)-FRLS 5х25мк(N,PE)-1 ТУ 2...",Черный,25мк,LS,ТПЖ 25мк,1910,Б-630



Барабаны ТПЖ под FR-обмотку (нанести обмотку на отмеченные участки):
  ТПЖ 25мк        барабан ТПЖ 25мк              → 17525 м FR-обмотки


In [9]:
# ─── Выходные барабаны ───────────────────────────────────────────────
rows = []
for da in result.drum_assignments:
    rows.append({
        'ID':         da.id,
        'Марка':      da.cable_mark,
        'Тип':        da.drum_type,
        'Ёмкость, м': int(da.drum_capacity),
        'Отрезки':    ', '.join(str(int(s)) for s in da.segments),
        'Итого, м':   int(da.total_length),
        'Загрузка':   f"{da.total_length/da.drum_capacity*100:.0f}%" if da.drum_capacity else '—',
        'Источник':   da.source,
    })
pd.DataFrame(rows)

,ID,Марка,Тип,"Ёмкость, м",Отрезки,"Итого, м",Загрузка,Источник
0,БК-001,"КОСМОГЕР Вз-ВБШвнг(А)-LS 3х2,5ок(N,PE)-1 ТУ 2...",№10,600,465,465,78%,партия
1,БК-002,"КОСМОГЕР Вз-ВБШвнг(А)-LS 3х25мк(N,PE)-1 ТУ 27...",№14,980,980,980,100%,партия
2,БК-003,"КОСМОГЕР Вз-ВБШвнг(А)-LS 3х25мк(N,PE)-1 ТУ 27...",№14,980,720,720,73%,партия
3,БК-004,КОСМОГЕР Вз-ВБШвнг(А)-LS 4х16мк(N)-1 ТУ 27.32...,№14,950,950,950,100%,партия
4,БК-005,КОСМОГЕР Вз-ВБШвнг(А)-LS 4х16мк(N)-1 ТУ 27.32...,№14,950,950,950,100%,партия
...,...,...,...,...,...,...,...,...
91,БК-092,"КОСМОСИЛ-НН КРэВЭнг(А)-LS 7х1,5-0,66 ТУ 27.32....",№14,800,700,700,88%,партия
92,БК-093,"КОСМОСИЛ-НН КРэВЭнг(А)-LS 7х1,5-0,66 ТУ 27.32....",№14,800,800,800,100%,партия
93,БК-094,"КОСМОСИЛ-НН КРэВЭнг(А)-LS 7х1,5-0,66 ТУ 27.32....",№14,800,700,700,88%,партия
94,БК-095,"КОСМОСИЛ-НН РэВнг(А)-LS 3х6ок(N,PE)-1 ТУ 27.32...",№14,800,800,800,100%,партия


In [10]:
# ─── Экспорт ─────────────────────────────────────────────────────────
export(OUTPUT_FILE, result)
print('Готово!')

Результат сохранён: output26.xlsx
Готово!
